# notebook card manifest v1

目标：验证 `research/<experiment>/` 与 `production/<strategy>/` 的 notebook-first 证据面可以通过稳定 `src.core` 接口扫描、汇总并写出 manifest，避免只靠人工浏览目录判断。

边界：notebook 只编排扫描和断言；card 扫描、主 notebook 判定、中文结论字段检查和 sidecar 写出逻辑必须在 `src.core`。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists() and (candidate / 'AGENTS.md').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core import scan_notebook_cards, write_notebook_card_manifest

CARD = Path('research/notebook_card_manifest_v1')
CARD.mkdir(parents=True, exist_ok=True)
CARD


PosixPath('research/notebook_card_manifest_v1')

In [2]:
scan = scan_notebook_cards(REPO_ROOT)
write_result = write_notebook_card_manifest(CARD, scan)
manifest = json.loads(Path(write_result['manifest_path']).read_text(encoding='utf-8'))
summary = manifest['summary']
summary


{'canonical_notebook_count': 91,
 'card_count': 91,
 'chinese_conclusion_ok_count': 91,
 'missing_chinese_conclusion_count': 0,
 'missing_notebook_count': 0,
 'multiple_notebook_count': 0,
 'noncanonical_notebook_count': 0,
 'notebook_count': 91,
 'production_card_count': 2,
 'production_manifest_missing_count': 0,
 'production_published_count': 1,
 'production_registry_mismatch_count': 0,
 'production_template_count': 1,
 'ready_count': 91,
 'schema_version': 'core.notebook_card_scan.summary.v1',
 'status_counts': {'ready': 91},
 'surface_count': 2}

In [3]:
noncanonical_cards = [
    {
        'surface': card['surface'],
        'card_id': card['card_id'],
        'primary_notebook': card['primary_notebook'],
        'canonical_notebook_name': card['canonical_notebook_name'],
        'status': card['status'],
    }
    for card in manifest['cards']
    if card['status'] == 'noncanonical_notebook_name'
]

analysis_summary = {
    'schema_version': 'research.notebook_card_manifest.v1',
    'status': 'completed',
    'card_count': summary['card_count'],
    'notebook_count': summary['notebook_count'],
    'missing_notebook_count': summary['missing_notebook_count'],
    'multiple_notebook_count': summary['multiple_notebook_count'],
    'missing_chinese_conclusion_count': summary['missing_chinese_conclusion_count'],
    'canonical_notebook_count': summary['canonical_notebook_count'],
    'noncanonical_notebook_count': summary['noncanonical_notebook_count'],
    'chinese_conclusion_ok_count': summary['chinese_conclusion_ok_count'],
    'production_card_count': summary['production_card_count'],
    'production_published_count': summary['production_published_count'],
    'production_template_count': summary['production_template_count'],
    'production_manifest_missing_count': summary['production_manifest_missing_count'],
    'production_registry_mismatch_count': summary['production_registry_mismatch_count'],
    'status_counts': summary['status_counts'],
    'noncanonical_cards': noncanonical_cards,
    'manifest_hash': manifest['manifest_hash'],
    'artifacts': {
        'manifest': Path(write_result['manifest_path']).name,
        'csv': Path(write_result['csv_path']).name,
        'review': Path(write_result['review_path']).name,
    },
}

assert analysis_summary['card_count'] > 0
assert analysis_summary['card_count'] == analysis_summary['notebook_count']
assert analysis_summary['missing_notebook_count'] == 0
assert analysis_summary['multiple_notebook_count'] == 0
assert analysis_summary['missing_chinese_conclusion_count'] == 0
assert analysis_summary['chinese_conclusion_ok_count'] == analysis_summary['card_count']
assert analysis_summary['production_published_count'] == 1
assert analysis_summary['production_template_count'] == 1
assert analysis_summary['production_manifest_missing_count'] == 0
assert analysis_summary['production_registry_mismatch_count'] == 0
assert len(analysis_summary['manifest_hash']) == 64

(CARD / 'analysis_summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_summary


{'schema_version': 'research.notebook_card_manifest.v1',
 'status': 'completed',
 'card_count': 91,
 'notebook_count': 91,
 'missing_notebook_count': 0,
 'multiple_notebook_count': 0,
 'missing_chinese_conclusion_count': 0,
 'canonical_notebook_count': 91,
 'noncanonical_notebook_count': 0,
 'chinese_conclusion_ok_count': 91,
 'production_card_count': 2,
 'production_published_count': 1,
 'production_template_count': 1,
 'production_manifest_missing_count': 0,
 'production_registry_mismatch_count': 0,
 'status_counts': {'ready': 91},
 'noncanonical_cards': [],
 'manifest_hash': '3bc47d41de1d6e0ed2d01a4d304ab07c093890380092e37616332597379ebc2c',
 'artifacts': {'manifest': 'notebook_card_manifest.json',
  'csv': 'notebook_cards.csv',
  'review': 'notebook_card_review.md'}}

## 中文结论

- 做了什么：新增并验证 notebook card manifest 流程，用 `src.core.scan_notebook_cards` 扫描 `research/` 与 `production/`，再用 `write_notebook_card_manifest` 写出 `notebook_card_manifest.json`、`notebook_cards.csv` 和中文 review。
- 核心指标：当前扫描要求每个 card 有且只有一个主 notebook，且最后中文结论包含做了什么、核心指标、regime 差异、是否晋升、风险、下一步；manifest hash 用于后续 review 对比；production card 额外记录 published/template、manifest present、registry registered 和 manifest hash match 状态。
- regime 差异：本实验不重新评估市场 regime；它验证每个研究/生产 notebook 都能承载 regime 结论字段，具体 regime 结果仍由各自 analysis sidecar 承载。
- 是否晋升：否。本 notebook 只验证证据面治理契约，不晋升任何模型或策略。
- 风险：scanner 只能检查 notebook 结构、结论字段和 production manifest/registry 连接是否存在，不能判断结论质量或策略未来表现；非 canonical notebook 命名仍需要人工 review 是否重命名。
- 下一步：把该 manifest 作为后续 schedule/dashboard 或架构审计的输入，逐步处理非 canonical notebook 命名、新增 card 和 production manifest 发布证据缺口。